En este cuaderno se revisan las pruebas estadísticas que determinan si una combinación de pesos es mejor con respecto a otra

In [11]:
from tqdm.notebook import tqdm as tqm
import matplotlib.pyplot as plt
import plotly.graph_objs as go
from utils.func_aux import *
from utils.func_vis import *
import plotly.express as px
import scipy.stats as st
import seaborn as sns
import pandas as pd
import numpy as np

# Opciones de matplotlib
rc = plt.rcParams
rc["figure.figsize"] = [15, 5]

# Para mostrar todas las columnas cuando se imprime un df
pd.set_option("display.max_columns", None)

# Para poner el estilo de las gráficas de matplotlib parecido al de ggplot
plt.style.use("ggplot") 

# Definimos tablas que usaremos en el cuaderno
df_todos_PI=pd.read_csv('../tablas_generadas/todos_QI.csv')
w_sorted=np.sort(df_todos_PI['w_0'].unique())

df_QI=pd.read_csv('../tablas_generadas/QI_carac.csv')
dicc_indicadores={df_QI.indicador[i]:df_QI.Meta[i] for i in range(len(df_QI))}

go_to_Assesment()


# Kruskal-Wallis

Usamos la prueba de Kruskal Wallis para ver si hay diferencia significativa entre cada una de las diferentes combinaciones de parámetros. 

Aquellos donde se rechaza la hiṕótesis nula presentan una media diferente y por lo tanto si dependen de esta elección.

In [12]:
# def get_KW(df):
#     indicador_lista=[]
#     for wi in df.w_0.unique():
#         df_w0_unico =df.query(f'(w_0=={wi})').sort_values('run')
#         indicador_lista.append(df_w0_unico.valor_indicador.values)
#     try:
#         return st.kruskal(*indicador_lista)[1]
#     except:
#         return np.nan

# # Tarda 28 segundos
# df_KW_todos=df_todos_PI.dropna().groupby(['n_objetivos','problema','indicador','hiperparam_ind_conv']).apply(get_KW).reset_index().rename(columns={0:'p-value_KW'})
# df_KW_todos['Medias_Distintas']=df_KW_todos['p-value_KW'].apply(lambda x: 1 if x <= 0.05 else 0) # Para que se puedan sumar los False y True
# df_KW_todos

# df_KW_todos=pd.read_csv('../tablas_generadas/KW_todos.csv')
# df_KW_todos.sample(5)

# Prueba de Friedman

La prueba de Kruskal se realiza con la suposición de que las muestras vienen de poblaciones no relacionadas. Dado que nuestros algoritmos distintos comparten puntos iniciales (las inicializaciones de las poblaciones dadas por las semillas) entonces tenemos que usar una prueba que compare a cada una de las corridas entre sí. 

Para esto, usamos la [prueba de Friedman](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.friedmanchisquare.html), obtenemos los $p$-valores y determinamos si se rechaza la hipótesis nula:

```H0: Todos los algoritmos tienen el mismo desempeño```

In [23]:
df_todos_PI = pd.read_csv('../tablas_generadas/QI_R2_EPSP_IGD+.csv')

In [24]:
df_todos_PI.query('(problema=="WFG9") & (n_objetivos==3) &(w_0==0.001) & (run==0) & (indicador=="eps+")')

,hiperparam_ind_conv,problema,n_objetivos,w_0,run,indicador,valor_indicador
0,IGD+,WFG9,3,0.001,0,eps+,0.217718
147840,EPS+,WFG9,3,0.001,0,eps+,0.179001
295680,R2,WFG9,3,0.001,0,eps+,0.221069


In [25]:
def get_Friedman(df):
    indicador_lista=[]
    for wi in df.w_0.unique():
        df_w0_unico =df.query(f'(w_0=={wi})').sort_values('run')
        indicador_lista.append(df_w0_unico.valor_indicador.values)
    try:
        return st.friedmanchisquare(*indicador_lista)[1]
    except:

        return np.nan

# Tarda 28 segundos
df_Friedman_todos=df_todos_PI.dropna().groupby(['n_objetivos','problema','indicador','hiperparam_ind_conv']).apply(get_Friedman).reset_index().rename(columns={0:'p-value_Friedman'})
df_Friedman_todos['Medias_Distintas']=df_Friedman_todos['p-value_Friedman'].apply(lambda x: 1 if x <= 0.05 else 0) # Para que se puedan sumar los False y True
# df_Friedman_todos =pd.read_csv('../tablas_generadas/Friedman_todos.csv')
df_Friedman_todos.sample(5,random_state=42)

c:\Users\fer_a\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\stats\_stats_py.py:8701: RuntimeWarning:

invalid value encountered in double_scalars



,n_objetivos,problema,indicador,hiperparam_ind_conv,p-value_Friedman,Medias_Distintas
1198,5,WFG3,eps+,IGD+,4.357209e-01,0
526,3,WFG3,eps+,IGD+,1.286938e-02,1
393,3,DTLZ3,s-energy,EPS+,1.771240e-30,1
1407,6,DTLZ4,eps+,EPS+,7.272890e-01,0
433,3,DTLZ5,r2,IGD+,4.688378e-01,0


In [26]:
df_Friedman_todos.to_csv('../tablas_generadas/Friedman_R2_EPS+_IGD+.csv',index=False)
df_Friedman_todos = pd.read_csv('../tablas_generadas/Friedman_R2_EPS+_IGD+.csv')

Creamos tablas de la clasificación principal de cada indicador de calidad

In [27]:
QI_carac=pd.read_csv('../tablas_generadas/QI_carac.csv')
QI_carac

,indicador,Categoría,Meta
0,igd+,Convergencia,Minimize
1,r2,Convergencia,Maximize
2,s-energy,Diversidad,Minimize
3,eps+,Convergencia,Minimize
4,igd,Convergencia,Minimize
5,spd,Diversidad,Maximize
6,hv,Convergencia,Maximize


In [28]:
ind_conv_lista=['IGD+','R2','EPS+']
for ind_conv in ind_conv_lista:
    df_Fried_ind=df_Friedman_todos.query(f'hiperparam_ind_conv=="{ind_conv}"')
    
    df_Fried_ind.n_objetivos=df_Fried_ind.n_objetivos.astype(str)
    df_Fried_ind=df_Fried_ind.merge(right=QI_carac,how='inner',on='indicador')
    df_Fried_ind = df_Fried_ind.rename(columns={'n_objetivos': 'n_obj','problema':'problem','indicador':'QI','hiperparam_ind_conv':"DE_conv_ind","p-value_Friedman":"p-value"})
    fig=px.bar(data_frame=df_Fried_ind.groupby(['n_obj','QI']).Medias_Distintas.mean().reset_index(),x='n_obj',y='Medias_Distintas',color='QI', barmode='group')
    fig.update_layout(
        xaxis_title='n obj', yaxis_title='Avg Differences', yaxis=dict(range=[0,1]),
        height=500, width=900
    )
    fig.layout.yaxis.tickformat = ',.0%' 
    fig.update_xaxes(categoryorder='array',categoryarray=['2', '3','4', '5','6', '7', '10']) 
    fig.write_image(f'../imgs_pdf/Friedman_obj_indconv_{ind_conv}.pdf')
    fig.show()

C:\Users\fer_a\AppData\Local\Temp\ipykernel_8604\1844519829.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



C:\Users\fer_a\AppData\Local\Temp\ipykernel_8604\1844519829.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



C:\Users\fer_a\AppData\Local\Temp\ipykernel_8604\1844519829.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



Para igd+ en 3 dimensiones casi todas las dimensiones sufren un cambio cuando se cambia $w_0"

In [29]:
QI_carac.replace({'Convergencia':'Convergence','Diversidad':'Diversity'})

,indicador,Categoría,Meta
0,igd+,Convergence,Minimize
1,r2,Convergence,Maximize
2,s-energy,Diversity,Minimize
3,eps+,Convergence,Minimize
4,igd,Convergence,Minimize
5,spd,Diversity,Maximize
6,hv,Convergence,Maximize


In [30]:
for ind_conv in ind_conv_lista:
    df_Friedman_ind=df_Friedman_todos.query(f'hiperparam_ind_conv=="{ind_conv}"')
    df_Friedman_ind=df_Friedman_ind.merge(QI_carac,on='indicador')
    df_Friedman_ind=df_Friedman_ind.rename(columns={'Categoría':'QI_cat'}).replace({'Convergencia':'Convergence','Diversidad':'Diversity'})
    fig=px.bar(data_frame=df_Friedman_ind.groupby(['n_objetivos','QI_cat']).Medias_Distintas.mean().reset_index(),x='n_objetivos',y='Medias_Distintas',color='QI_cat', barmode='group')
    fig.update_layout(
        # title=f'Proporción de diferencias por categoría. indicador {ind_conv}',
        xaxis_title='n obj', yaxis_title='Avg Differences', yaxis=dict(range=[0,1]),
        height=500, width=900)
    fig.layout.yaxis.tickformat = ',.0%' 
    fig.update_xaxes(categoryorder='array',categoryarray=['2', '3', '4','5','6', '7', '10']) 
    fig.write_image(f'../imgs_pdf/Friedman_Diferencia_por_categoria_{ind_conv}.pdf')
    fig.show()

Hay que ver si estas diferencias se deben a los indicadores que no están

# Comparación uno a uno

## Wilcoxon

En el siguiente código se calcula el valor $p$ de la prueba intercalando para los indicadores que se quieren maximizar y los que se quieren minimizar.

Esto se guarda en un dataframe que se lee para cada llave como el valor p de la prueba estadística de que los datos i no son mejores que los datos j. Así, cuando este valor sea menor a 0.05 podremos decir que de manera significativa el renglón es mejor a la columna para cada indicador

Hacemos para el problema con los dos tipos de indicadores de convergencia `R2,IGD+` al mismo tiempo

In [39]:
w_sorted = df_todos_PI['w_0'].unique()

In [46]:
df_todos_PI['valor_indicador']  = df_todos_PI['valor_indicador'].astype('float')

In [47]:
df_todos_PI

,hiperparam_ind_conv,problema,n_objetivos,w_0,run,indicador,valor_indicador
0,IGD+,WFG9,3,0.001,0,eps+,0.217718
1,IGD+,WFG9,3,0.001,1,eps+,0.240986
2,IGD+,WFG9,3,0.001,2,eps+,0.240616
3,IGD+,WFG9,3,0.001,3,eps+,0.236098
4,IGD+,WFG9,3,0.001,4,eps+,0.267390
...,...,...,...,...,...,...,...
443515,R2,WFG9,6,0.999,15,hv,94075.890000
443516,R2,WFG9,6,0.999,16,hv,105533.000000
443517,R2,WFG9,6,0.999,17,hv,91050.030000
443518,R2,WFG9,6,0.999,18,hv,95307.580000


In [54]:
df_todos_PI_parte = df_todos_PI.query('	(problema=="WFG9") &	(n_objetivos==3)	& (indicador=="eps+")')
df_todos_PI_parte

,hiperparam_ind_conv,problema,n_objetivos,w_0,run,indicador,valor_indicador
0,IGD+,WFG9,3,0.001,0,eps+,0.217718
1,IGD+,WFG9,3,0.001,1,eps+,0.240986
2,IGD+,WFG9,3,0.001,2,eps+,0.240616
3,IGD+,WFG9,3,0.001,3,eps+,0.236098
4,IGD+,WFG9,3,0.001,4,eps+,0.267390
...,...,...,...,...,...,...,...
430095,R2,WFG9,3,0.999,15,eps+,0.259823
430096,R2,WFG9,3,0.999,16,eps+,0.248132
430097,R2,WFG9,3,0.999,17,eps+,0.244226
430098,R2,WFG9,3,0.999,18,eps+,0.222618


In [56]:
def WC_apply_all(df):
    _,_,indicador,_ = df.name
    mat_WC=np.zeros(shape=(len(w_sorted),len(w_sorted)))
    for i,wi in enumerate(w_sorted):
        for j,wj in enumerate(w_sorted):
            datos_i=df.query(f'w_0=={wi}').valor_indicador.values
            datos_j=df.query(f'w_0=={wj}').valor_indicador.values
            # print()
            # print(datos_i)
            # print(wj,datos_j)

            if (datos_i==datos_j).all():
                mat_WC[i,j]=np.nan
            elif dicc_indicadores[indicador]=='Maximize':
                try:
                    mat_WC[i,j]=st.wilcoxon(datos_i,datos_j,alternative='greater')[1]
                except:
                    mat_WC[i,j]=np.nan
            
            elif dicc_indicadores[indicador]=='Minimize':
                try:
                    mat_WC[i,j]=st.wilcoxon(datos_i,datos_j,alternative='less')[1]
                except:
                    mat_WC[i,j]=np.nan
        
    return pd.DataFrame(mat_WC,index=w_sorted,columns=w_sorted)

# Tarda como 10 minutos en acabar
df_WC_todos=df_todos_PI.groupby(['n_objetivos','problema','indicador','hiperparam_ind_conv']).apply(WC_apply_all)
df_WC_todos_exp=df_WC_todos.applymap(format_scientific_6_digits)
# df_WC_todos_exp.to_csv('../tablas_generadas/WC_QI__R2_EPS+_IGD+.csv')
# df_WC_todos=pd.read_csv('../tablas_generadas/WC_QI__R2_EPS+_IGD+.csv').drop(columns=['Unnamed: 4']).set_index(['n_objetivos','problema','indicador','hiperparam_ind_conv'])
df_WC_todos

c:\Users\fer_a\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\stats\_morestats.py:3414: UserWarning:

Exact p-value calculation does not work if there are zeros. Switching to normal approximation.

c:\Users\fer_a\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\stats\_morestats.py:3414: UserWarning:

Exact p-value calculation does not work if there are zeros. Switching to normal approximation.

c:\Users\fer_a\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\stats\_morestats.py:3414: UserWarning:

Exact p-value calculation does not work if there are zeros. Switching to normal approximation.

c:\Users\fer_a\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\stats\_morestats.py:3414: UserWarning:

Exact p-value calculation does not work if there are zeros. Switching to normal approximation.

c:\Users\fer_a\AppData\Local\Programs\Python\Python310\lib\site-packages\scipy\stats\_morestats.py:3414: UserWarning:

Exact p-value calcula

0.001     0.100  \
n_objetivos problema indicador hiperparam_ind_conv                             
2           DTLZ1    eps+      EPS+                0.001       NaN  0.285298   
                                                   0.100  0.727062       NaN   
                                                   0.200  0.434744  0.204549   
                                                   0.300  0.740172  0.449159   
                                                   0.400  0.990383  0.905326   
...                                                            ...       ...   
7           WFG9     spd       R2                  0.600  0.500000  0.168144   
                                                   0.700  0.158655  0.029391   
                                                   0.800  0.760250  0.369441   
                                                   0.900  0.327360  0.051235   
                                                   0.999  0.327360  0.078650   

                                                             0.200     0.300  \
n_objetivos problema indicador hiperparam_ind_conv                             
2           DTLZ1    eps+      EPS+                0.001  0.579589  0.259828   
                                                   0.100  0.805812  0.565256   
                                                   0.200       NaN  0.147126   
                                                   0.300  0.861322       NaN   
                                                   0.400  0.983616  0.951346   
...                                                            ...       ...   
7           WFG9     spd       R2                  0.600  0.158655  0.007558   
                                                   0.700  0.016947  0.002322   
                                                   0.800  0.207108  0.009900   
                                                   0.900  0.047790  0.002335   
                                                   0.999  0.047790  0.002404   

                                                             0.400     0.500  \
n_objetivos problema indicador hiperparam_ind_conv                             
2           DTLZ1    eps+      EPS+                0.001  0.010742  0.006808   
                                                   0.100  0.101225  0.076823   
                                                   0.200  0.018117  0.014788   
                                                   0.300  0.052699  0.082479   
                                                   0.400       NaN  0.536361   
...                                                            ...       ...   
7           WFG9     spd       R2                  0.600  0.089856  0.158655   
                                                   0.700  0.016947  0.007153   
                                                   0.800  0.281851  0.239750   
                                                   0.900  0.047790  0.012674   
                                                   0.999  0.047790  0.029391   

                                                             0.600     0.700  \
n_objetivos problema indicador hiperparam_ind_conv                             
2           DTLZ1    eps+      EPS+                0.001  0.026585  0.008591   
                                                   0.100  0.048654  0.088427   
                                                   0.200  0.034790  0.011975   
                                                   0.300  0.048654  0.037926   
                                                   0.400  0.449159  0.662889   
...                                                            ...       ...   
7           WFG9     spd       R2                  0.600       NaN  0.910144   
                                                   0.700  0.089856       NaN   
                                                   0.800  0.630559  0.977250   
                                                   0.900  

In [57]:
df_WC_todos_exp.to_csv('../tablas_generadas/WC_QI__R2_EPS+_IGD+.csv')

## Heatmaps

Se hace una función para calcular un heatmap por cada problema, dimensión e indicador

In [26]:
# df_WC_todos = pd.read_csv('../tablas_generadas/WC_QI_todos.csv').drop(columns=['Unnamed: 4']).set_index(['n_objetivos','problema','indicador','hiperparam_ind_conv'])

In [59]:

# def get_heatmap(df_WC, df_todos_PI, problema, n_objetivos, indicador,ind_conv='IGD+'):
#     """Regresa el heatmap de wilcoxon, el boxplot para ver que efectivamente uno le está ganando al otro y un dataframe con la media para ver si coinciden los datos"""
#     plt.figure(figsize=(10,5))
#     # plt.title(
#     #     f"$p$-value Wilcoxon. {problema} {n_objetivos} objetivos  {indicador.upper()}\n Color $\\rightarrow$ renglón mejor que columna"
#     # )
#     mask = df_WC.loc[(n_objetivos, problema, indicador.lower())].astype(float) < 0.05
#     fig, ax = plt.subplots(figsize=(10, 5))
#     sns.heatmap(
#         df_WC.loc[(n_objetivos, problema, indicador.lower())].astype(float),
#         vmin=0,
#         vmax=0.05,
#         mask=~mask,
#         ax=ax
#     )
#     ax.set_xlabel("$w_0$")
#     ax.set_ylabel("$w_0$")
#     # Set custom x-ticks and labels
#     ax.set_xticks(np.arange(len(w_0)) + 0.5)
#     ax.set_xticklabels([round(w0i[0], 2) for w0i in w_0], rotation=45)

#     # Set custom y-ticks and labels
#     ax.set_yticks(np.arange(len(w_0)) + 0.5)
#     ax.set_yticklabels([round(w0i[0], 2) for w0i in w_0])


#     plt.savefig(f'../imgs_pdf/heatmaps/Heatmap_{problema}_obj{n_objetivos}_{indicador}_indconv_{ind_conv}.pdf')
#     # plt.show()
#     plt.close()
#     get_boxplot(problema, n_objetivos, indicador, df_PI=df_todos_PI,save_img_path=f'../imgs_pdf/Box_{problema}_obj{n_objetivos}_{indicador}_indconv_{ind_conv}.pdf')
#     return (
#         df_todos_PI[
#             (df_todos_PI["problema"] == problema)
#             & (df_todos_PI["n_objetivos"] == n_objetivos)
#             & (df_todos_PI["indicador"] == indicador.lower())
#         ]
#         .groupby(["w_0"])
#         .valor_indicador.mean()
#         .reset_index()
#     )


# #* Para guardar las imágenes como pdf
# for hiperparam_ind_conv in df_todos_PI.hiperparam_ind_conv.unique():
#     for prob in df_todos_PI.problema.unique():
#         for n_obj in df_todos_PI.n_objetivos.unique():
#             for ind in df_todos_PI.indicador.unique():
#                 get_heatmap(df_WC_todos, df_todos_PI, prob, n_obj, ind, hiperparam_ind_conv)

## Conteo de borda

Se define como el número de victorias que tuvo sobre los demás

In [68]:
df_todos_PI.query('hiperparam_ind_conv=="R2"')[df_todos_PI['problema'].str.startswith('DTLZ')]

C:\Users\fer_a\AppData\Local\Temp\ipykernel_8604\2497015360.py:1: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



,hiperparam_ind_conv,problema,n_objetivos,w_0,run,indicador,valor_indicador
295720,R2,DTLZ4,4,0.001,0,hv,12.835320
295721,R2,DTLZ4,4,0.001,1,hv,14.820320
295722,R2,DTLZ4,4,0.001,2,hv,14.825200
295723,R2,DTLZ4,4,0.001,3,hv,14.783510
295724,R2,DTLZ4,4,0.001,4,hv,15.555100
...,...,...,...,...,...,...,...
443455,R2,DTLZ6,3,0.999,15,spd,8.824659
443456,R2,DTLZ6,3,0.999,16,spd,8.824676
443457,R2,DTLZ6,3,0.999,17,spd,8.824357
443458,R2,DTLZ6,3,0.999,18,spd,8.824752


In [60]:
def conteo_borda(df):
    victorias=(df<0.05).sum(axis=1).reset_index()
    return pd.DataFrame({'w_0':victorias.index, 'victorias':victorias[0]})
# df_WC_todos = df_WC_todos.drop(columns='Unnamed: 4')
df_borda=df_WC_todos.reset_index().groupby(['n_objetivos','problema','indicador','hiperparam_ind_conv']).apply(conteo_borda).reset_index().drop(columns='level_4')
# df_borda=pd.read_csv('../tablas_generadas/conteo_borda_todos.csv').set_index(['n_objetivos','problema','indicador','hiperparam_ind_conv'])
df_borda=df_borda.pivot(values='victorias',index=['n_objetivos','problema','indicador','hiperparam_ind_conv'],columns='w_0')
df_borda.to_csv('../tablas_generadas/conteo_borda__R2_EPS+_IGD+.csv')
df_borda

w_0                                                 0   1   2   3   4   5   \
n_objetivos problema indicador hiperparam_ind_conv                           
2           DTLZ1    eps+      EPS+                  7   2   5   2   0   0   
                               IGD+                  1   0   0   0   0   0   
                               R2                    8   5   5   5   0   0   
                     hv        EPS+                  1   0   0   0   0   0   
                               IGD+                  1   0   0   0   0   1   
...                                                 ..  ..  ..  ..  ..  ..   
7           WFG9     s-energy  IGD+                 11   7   6   5   4   0   
                               R2                    5   1   1   0   1   1   
                     spd       EPS+                  4   4   1   1   2   0   
                               IGD+                  8   1   1   0   4   0   
                               R2                    2   1   1   0   1   1   

w_0                                                 6   7   8   9   10  
n_objetivos problema indicador hiperparam_ind_conv                      
2           DTLZ1    eps+      EPS+                  0   0   0   0   0  
                               IGD+                  1   0   0   0   0  
                               R2                    0   0   0   7   5  
                     hv        EPS+                  0   0   0   0   0  
                               IGD+                  3   3   0   0   0  
...                                                 ..  ..  ..  ..  ..  
7           WFG9     s-energy  IGD+                  1   0   0   0   0  
                               R2                    2   4   1   2   2  
                     spd       EPS+                  0   0   0   2   2  
                               IGD+                  0   0   0   0   0  
                               R2                    1   6   1   4   4  

[2016 rows x 11 columns]

In [44]:
def get_random_problem(random_state=42):
    DE_conv_ind, problem, n_obj, w_0,run,QI,value_QI = list(df_todos_PI.sample(1,random_state=random_state).iloc[0].values)
    return DE_conv_ind, problem, n_obj, QI 
    
DE_conv_ind, problem, n_obj, QI  = get_random_problem()
DE_conv_ind, problem, n_obj, QI

('IGD+', 'DTLZ7', 7, 'hv')

In [62]:
w_0 = get_w_espaciados(10)
def conteo_borda_problema(df_borda,n_objetivos,problema,indicador):
    df_vs=pd.melt(df_borda.loc[n_objetivos,problema,indicador].reset_index(),id_vars=['DE_conv_ind'])

    df_vs['w_0']=df_vs['w_0'].replace({i:w_0[i][0] for i in range(len(w_0))})
    fig=px.bar(data_frame=df_vs,x='w_0',y='value',color='DE_conv_ind',barmode='group')
    fig.update_layout(
        # title=f'Conteo de borda {problema} {n_objetivos}D {indicador.upper()}',
        xaxis_title='$w_0$', yaxis_title='Victories',
        height=500, width=900
    )
    fig.write_image(f'../imgs_pdf/conteo_borda_{problema}_obj{n_objetivos}_ind{indicador}.pdf')
    fig.show()
    return df_vs

df_borda= df_borda.reset_index().rename(columns={'hiperparam_ind_conv':'DE_conv_ind'}).set_index(['n_objetivos','problema','indicador','DE_conv_ind'])
n_objetivos,problema,indicador=3,'WFG4','hv'
df_vs=conteo_borda_problema(df_borda,n_objetivos,problema,indicador)

In [63]:
df_borda_stack=df_borda.groupby(['n_objetivos','DE_conv_ind'])[df_borda.columns].mean().unstack().reset_index()

for n_objetivos in df_borda_stack.n_objetivos.unique():
    df=df_borda_stack[df_borda_stack['n_objetivos']==n_objetivos].iloc[:,1:].unstack().reset_index().drop(columns=['level_2'])
    df['w_0']=df['w_0'].replace({f'{i}':w_0[i][0] for i in range(len(w_0))})
    fig=px.bar(df,x='w_0',y=0,color='DE_conv_ind',barmode='group')
    fig.update_layout(
        # title=f'Conteo de borda por peso.\nNúmero de objetivos {n_objetivos}',
        xaxis_title='$w_0$', yaxis_title='Avg Victories',
        height=400, width=600
    )
    fig.write_image(f'../imgs_pdf/borda_obj_{n_objetivos}.pdf')
    fig.show()

In [64]:
df_borda_stack=df_borda.groupby(['indicador','DE_conv_ind'])[df_borda.columns].mean().unstack().reset_index()

for indicador in df_borda_stack.indicador.unique():
    print(indicador)
    df=df_borda_stack[df_borda_stack['indicador']==indicador].iloc[:,1:].unstack().reset_index().drop(columns=['level_2'])
    df['w_0']=df['w_0'].replace({f'{i}':w_0[i][0] for i in range(len(w_0))})
 
    fig=px.bar(df,x='w_0',y=0,color='DE_conv_ind',barmode='group')
    fig.update_layout(
        # title=f'Conteo de borda por peso.\nindicador {indicador}',
        xaxis_title='$w_0$', yaxis_title='Avg Victories',
        height=400, width=600
    )
    fig.write_image(f'../imgs_pdf/borda_obj_ind_{indicador}.pdf')

    fig.show()

eps+


hv


igd


igd+


r2


s-energy


spd
